# Lesson 09: Building a Web App for an ML Model (UFO Sightings)

Train a `LogisticRegression` classifier on self-contained NUFORC UFO sighting records, serialize it to `ufo-model.pkl` with `pickle`, generate a Flask `app.py`, and test an interactive **Gradio** web interface.

In [ ]:
!pip install -q gradio scikit-learn pandas

In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(42)
centers = {'au': (-25, 134), 'ca': (56, -106), 'de': (51, 10), 'gb': (54, -2), 'us': (39, -98)}
rows = []
for code, (lat, lon) in centers.items():
    for _ in range(60):
        rows.append({
            'Seconds': int(rng.integers(2, 58)),
            'Country': code,
            'Latitude': lat + rng.normal(0, 4),
            'Longitude': lon + rng.normal(0, 6),
        })
ufos = pd.DataFrame(rows)
le = LabelEncoder()
ufos['Country'] = le.fit_transform(ufos['Country'])

X = ufos[['Seconds', 'Latitude', 'Longitude']]
y = ufos['Country']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

model = LogisticRegression(max_iter=500).fit(X_train, y_train)
print(f"Test Accuracy: {accuracy_score(y_test, model.predict(X_test)) * 100:.1f}%")

with open('ufo-model.pkl', 'wb') as f:
    pickle.dump(model, f)

reloaded = pickle.load(open('ufo-model.pkl', 'rb'))
countries = ['Australia', 'Canada', 'Germany', 'UK', 'US']
pred_idx = reloaded.predict([[50, 44, -12]])[0]
print('Prediction for [50, 44, -12]:', countries[pred_idx])

In [ ]:
import gradio as gr

def predict_ufo_country(seconds, latitude, longitude):
    idx = reloaded.predict([[seconds, latitude, longitude]])[0]
    return f"Likely country: {countries[idx]}"

demo = gr.Interface(
    fn=predict_ufo_country,
    inputs=[gr.Number(value=50, label='Seconds'), gr.Number(value=44, label='Latitude'), gr.Number(value=-12, label='Longitude')],
    outputs='text',
    title='UFO Sighting Country Predictor'
)
demo.launch(share=False)

## Assignment: Try a Different Model

Replace `LogisticRegression` with `RandomForestClassifier` or `SVC`, re-pickle to `ufo-model.pkl`, and compare test accuracy.